In [2]:
!git clone https://github.com/mode-str/crossview
%cd crossview

Cloning into 'crossview'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 111 (delta 7), reused 0 (delta 0), pack-reused 96 (from 1)
Receiving objects: 100% (111/111), 77.62 KiB | 7.76 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/crossview


In [3]:
!pip install -r requirement.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 19.0 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26


In [4]:
!gdown 1UyVyFJ_pRaJHIr_eBY2HL7gkS5y9UxqI -O /content/SUES-200.zip

Downloading...
From (original): https://drive.google.com/uc?id=1UyVyFJ_pRaJHIr_eBY2HL7gkS5y9UxqI
From (redirected): https://drive.google.com/uc?id=1UyVyFJ_pRaJHIr_eBY2HL7gkS5y9UxqI&confirm=t&uuid=639d60de-f4a3-49ec-9478-119d7b8a856b
To: /content/SUES-200.zip
100% 5.70G/5.70G [01:01<00:00, 92.8MB/s]


In [5]:
!unzip -nq /content/SUES-200.zip -d /content/

In [6]:
!ls -la /content/SUES-200-512x512

total 16
drwxr-xr-x   4 root root 4096 Dec  7  2022 .
drwxr-xr-x   1 root root 4096 Apr  6 07:54 ..
drwxr-xr-x 202 root root 4096 Dec  7  2022 drone_view_512
drwxr-xr-x 202 root root 4096 Apr 23  2022 satellite-view


Исправим структуру train датасета.

Наша структура:

```
SUES-200-512x512/
└── drone_view_512/
    ├── 0001/
    │   ├── 150/
    │   │   └── 0.jpg
    │   └── 200/
    └── ...
```



Требуемая структура:

```
SUES-200/
├── train/
│   ├── drone/
│   │   ├── 0001/
│   │   │   ├── 0.jpg
│   │   │   └── ...
│   └── satellite/
│       └── ...
└── test/
    └── ...
```



In [7]:
import os
import shutil
from pathlib import Path

In [8]:
source_root = "/content/SUES-200-512x512"
target_root = "/content/SUES-200"

In [9]:
drone_path = os.path.join(source_root, "drone_view_512")
all_buildings = [b for b in os.listdir(drone_path) if os.path.isdir(os.path.join(drone_path, b))]
all_buildings_sorted = sorted(all_buildings, key=lambda x: int(x))
test_buildings = all_buildings_sorted[:40]
train_buildings = all_buildings_sorted[40:]

In [10]:
def get_split(building_id):
    return 'test' if building_id in test_buildings else 'train'

In [11]:
for view in ['drone', 'satellite']:
    for building in train_buildings:
        os.makedirs(f"{target_root}/train/{view}/{building}", exist_ok=True)
    for building in test_buildings:
        os.makedirs(f"{target_root}/test/{view}/{building}", exist_ok=True)

In [12]:
if os.path.exists(drone_path):
    print("Processing drone_view_512")

    # Проходим по всем папкам зданий (0001, 0002, etc.)
    for building in os.listdir(drone_path):
        building_path = os.path.join(drone_path, building)
        if not os.path.isdir(building_path):
            continue

        split = get_split(building)

        # Проходим по высотам (150, 200, 250, 300)
        for altitude in os.listdir(building_path):
            altitude_path = os.path.join(building_path, altitude)
            if not os.path.isdir(altitude_path):
                continue

            # Копируем drone изображения
            drone_images = os.listdir(altitude_path)
            for img_file in drone_images:
                if img_file.endswith('.jpg'):
                    src = os.path.join(altitude_path, img_file)
                    dst = os.path.join(target_root, f"{split}/drone/{building}/{altitude}_{img_file}")
                    shutil.copy2(src, dst)

    print("Drone-view done")
else:
    print(f"{drone_path} not found")

Processing drone_view_512
Drone-view done


In [14]:
sat_path = os.path.join(source_root, "satellite-view")
if os.path.exists(sat_path):
    print("Processing satellite-view")
    for building in os.listdir(sat_path):
        building_path = os.path.join(sat_path, building)
        if not os.path.isdir(building_path):
            continue

        split = get_split(building)

        # Находим satellite изображение
        src = os.path.join(building_path, "0.png")
        dst = os.path.join(target_root, f"{split}/satellite/{building}/0.png")
        shutil.copy2(src, dst)
    print("Satellite-view done")
else:
    print(f"{sat_path} not found")

Processing satellite-view
Satellite-view done


Теперь исправим структуру test датасета, она должна иметь вид:
```
test/
├── gallery_satellite/  
│   ├── 1001/
│   │   ├── img1.jpg
│   │   ├── img2.jpg
│   │   └── ...
│   ├── 1002/
│   │   └── ...
│   └── ...
├── gallery_drone/
│   ├── 1001/
│   │   └── ...
│   └── ...
├── query_satellite/
│   ├── 1001/
│   │   └── ...
│   └── ...
└── query_drone/
    ├── 1001/
    │   └── ...
    └── ...
```
При этом query и gallery не должны содержать одинаковые изображения.

In [36]:
!rm -r /content/SUES-200/test

In [37]:
source_drone = "/content/SUES-200-512x512/drone_view_512"
source_sat = "/content/SUES-200-512x512/satellite-view"
target = "/content/SUES-200/test"

os.makedirs(target, exist_ok=True)

print(f"Drone path exists: {os.path.exists(source_drone)}")
print(f"Satellite path exists: {os.path.exists(source_sat)}")

def copy_drone_images(src_building_dir, dst_building_dir):
    os.makedirs(dst_building_dir, exist_ok=True)
    for altitude in os.listdir(src_building_dir):
        alt_path = os.path.join(src_building_dir, altitude)
        if not os.path.isdir(alt_path):
            continue
        for file in os.listdir(alt_path):
            if file.endswith('.jpg'):
                src = os.path.join(alt_path, file)
                dst = os.path.join(dst_building_dir, f"{altitude}_{file}")
                if os.path.exists(src):
                    shutil.copy2(src, dst)
                else:
                    print(f"Warning: {src} not found")

def copy_satellite_images(src_building_dir, dst_building_dir):
    os.makedirs(dst_building_dir, exist_ok=True)
    src = os.path.join(src_building_dir, "0.png")
    dst = os.path.join(dst_building_dir, "0.png")

    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        print(f"Warning: {src} not found")

# Создаём все четыре подпапки (gallery_satellite, gallery_drone, query_satellite, query_drone)
for sub in ['gallery_satellite', 'gallery_drone', 'query_satellite', 'query_drone']:
    os.makedirs(os.path.join(target, sub), exist_ok=True)

# Обрабатываем спутниковые снимки (кладутся в gallery_satellite и query_satellite)
if os.path.exists(source_sat):
    for building in os.listdir(source_sat):
        if building in test_buildings:
            src_building = os.path.join(source_sat, building)
            if not os.path.isdir(src_building):
                continue
            # Копируем в gallery_satellite
            copy_satellite_images(src_building, os.path.join(target, 'gallery_satellite', building))
            # Копируем в query_satellite
            copy_satellite_images(src_building, os.path.join(target, 'query_satellite', building))

# Обрабатываем снимки дрона (кладутся в gallery_drone и query_drone)
if os.path.exists(source_drone):
    for building in os.listdir(source_drone):
        if building in test_buildings:
            src_building = os.path.join(source_drone, building)
            if not os.path.isdir(src_building):
                continue
            # Копируем в gallery_drone
            copy_drone_images(src_building, os.path.join(target, 'gallery_drone', building))
            # Копируем в query_drone
            copy_drone_images(src_building, os.path.join(target, 'query_drone', building))

Drone path exists: True
Satellite path exists: True


In [15]:
# !rm -r /content/SUES-200-512x512/

In [16]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [17]:
with open('datasets/__init__.py', 'w') as f:
    f.write('''
from .make_dataloader import make_dataset
from .queryDataset import Dataset_query, Query_transforms
from .Dataloader_University import *
from .autoaugment import *
from .random_erasing import *
''')

In [18]:
with open('losses/__init__.py', 'w') as f:
    f.write('''
from .cal_loss import *
from .triplet_loss import *
''')

In [19]:
import re

files_to_fix = [
    'datasets/autoaugment.py',
    'datasets/random_erasing.py',
    'datasets/make_dataloader.py'
]

for file_path in files_to_fix:
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            content = f.read()

        # Заменяем все устаревшие псевдонимы
        replacements = {
            'np.int': 'int',
            'np.float': 'float',
            'np.bool': 'bool',
            'np.complex': 'complex',
            'np.object': 'object',
            'np.str': 'str'
        }

        for old, new in replacements.items():
            # Заменяем только как отдельное слово или с атрибутами
            content = re.sub(r'\b' + old + r'\b', new, content)
            content = re.sub(r'\.astype\(' + old + r'\)', '.astype(' + new + ')', content)

        with open(file_path, 'w') as f:
            f.write(content)

        print(f"{file_path} fixed")

datasets/autoaugment.py fixed
datasets/random_erasing.py fixed
datasets/make_dataloader.py fixed


In [20]:
%%writefile datasets/Dataloader_University.py
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import Sampler
from torchvision import datasets, transforms
import os
import numpy as np
from PIL import Image

class Dataloader_University(Dataset):
    def __init__(self, root, transforms, names=['satellite', 'drone']):
        super(Dataloader_University, self).__init__()
        self.transforms_drone_street = transforms['train']
        self.transforms_satellite = transforms['satellite']
        self.root = root
        self.names = names

        # Собираем все ID зданий из первой доступной папки
        first_view = names[0]
        first_view_path = os.path.join(root, first_view)
        self.cls_names = [d for d in os.listdir(first_view_path) if os.path.isdir(os.path.join(first_view_path, d))]
        self.cls_names.sort(key=lambda x: int(x))  # сортируем по номеру

        # Создаем словарь с путями к изображениям
        self.dict_path = {}
        for name in names:
            dict_ = {}
            view_path = os.path.join(root, name)
            for cls_name in self.cls_names:
                cls_path = os.path.join(view_path, cls_name)
                if os.path.exists(cls_path):
                    img_list = [f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.png'))]
                    img_path_list = [os.path.join(cls_path, img) for img in img_list]
                    dict_[cls_name] = img_path_list
                else:
                    dict_[cls_name] = []
            self.dict_path[name] = dict_

        self.map_dict = {i: self.cls_names[i] for i in range(len(self.cls_names))}
        self.index_cls_nums = 2

    def sample_from_cls(self, name, cls_num):
        # Берем случайное изображение из папки здания
        img_paths = self.dict_path[name][cls_num]
        if not img_paths:
            # Если нет изображений этого типа, возвращаем черное изображение
            return Image.new('RGB', (256, 256), color='black')
        img_path = np.random.choice(img_paths, 1)[0]
        img = Image.open(img_path).convert('RGB')
        return img

    def __getitem__(self, index):
        cls_nums = self.map_dict[index]

        # Satellite изображение
        img = self.sample_from_cls("satellite", cls_nums)
        img_s = self.transforms_satellite(img)

        # Drone изображение (вместо street)
        img = self.sample_from_cls("drone", cls_nums)
        img_d = self.transforms_drone_street(img)

        # Для совместимости дублируем drone вместо img_st
        return img_s, img_d, img_d, index

    def __len__(self):
        return len(self.cls_names)


class Sampler_University(object):
    def __init__(self, data_source, batchsize=8, sample_num=4, triplet_loss=0):
        self.data_len = len(data_source)
        self.batchsize = batchsize
        self.sample_num = sample_num
        self.triplet_loss = triplet_loss

    def __iter__(self):
        list_ = np.arange(0, self.data_len)
        nums = np.repeat(list_, self.sample_num, axis=0)
        np.random.shuffle(nums)
        return iter(nums)

    def __len__(self):
        return len(self.data_source)


def train_collate_fn(batch):
    img_s, img_st, img_d, ids = zip(*batch)
    ids = torch.tensor(ids, dtype=torch.int64)
    return [torch.stack(img_s, dim=0), ids], [torch.stack(img_st, dim=0), ids], [torch.stack(img_d, dim=0), ids]


if __name__ == '__main__':
    transform_train_list = [
        transforms.Resize((256, 256), interpolation=3),
        transforms.Pad(10, padding_mode='edge'),
        transforms.RandomCrop((256, 256)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]

    transform_train_list = {
        "satellite": transforms.Compose(transform_train_list),
        "train": transforms.Compose(transform_train_list)
    }

    datasets = Dataloader_University(
        root="/content/SUES-200/train",
        transforms=transform_train_list,
        names=['satellite', 'drone']
    )

Overwriting datasets/Dataloader_University.py


In [ ]:
# !rm -r /content/crossview/model/
# !rm -r /content/SUES-200/

In [21]:
name = "convnext_tri"
data_dir = "/content/SUES-200/train"
test_dir = "/content/SUES-200/test"
gpu_ids = "0"
lr = 0.01
batchsize = 8
triplet_loss = 0.3
num_epochs = 200
views = 2

In [22]:
!python train.py \
    --name $name \
    --data_dir $data_dir \
    --gpu_ids $gpu_ids \
    --views $views \
    --lr $lr \
    --batchsize $batchsize \
    --triplet_loss $triplet_loss \
    --epochs $num_epochs

This is not an error. If you want to use low precision, i.e., fp16, please install the apex with cuda support (https://github.com/NVIDIA/apex) and update pytorch to 1.0
[AutoAugment ImageNet Policy, ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1), saturation=(0.9, 1.1), hue=None), Resize(size=(256, 256), interpolation=bicubic, max_size=None, antialias=True), Pad(padding=10, fill=0, padding_mode=edge), RandomCrop(size=(256, 256), padding=None), RandomHorizontalFlip(p=0.5), ToTensor(), Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), <datasets.random_erasing.RandomErasing object at 0x7d5771ad03e0>]
{'satellite': 160, 'street': 160, 'drone': 160}
===========building convnext===========
using model_type: convnext_tiny as a backbone
Downloading: "https://dl.fbaipublicfiles.com/convnext/convnext_tiny_22k_1k_224.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny_22k_1k_224.pth
100% 109M/109M [00:04<00:00, 23.3MB/s]
https://dl.fbaipublicfiles.com/convnext/convnext

In [38]:
!python test.py \
    --name $name \
    --test_dir $test_dir \
    --gpu_ids $gpu_ids \
    --mode 1

This is not an error. If you want to use low precision, i.e., fp16, please install the apex with cuda support (https://github.com/NVIDIA/apex) and update pytorch to 1.0
We use the scale: 1
-------test-----------
===========building convnext===========
using model_type: convnext_tiny as a backbone
https://dl.fbaipublicfiles.com/convnext/convnext_tiny_22k_1k_224.pth
Load the model from ./model/convnext_tri/net_152.pth
1 -> 3:
100% 5/5 [00:02<00:00,  2.27it/s]
100% 1000/1000 [03:00<00:00,  5.53it/s]
Test complete in 3m 3s
/content/crossview/evaluate.py:35: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  mask = np.in1d(index, junk_index, invert=True)
/content/crossview/evaluate.py:40: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  mask = np.in1d(index, good_index)
torch.Size([40, 1536])
torch.Size([8000, 1536])
80
Recall@1:100.00 Recall@5:100.00 Recall@10:100.00 Recall@top1:100.00 AP:97.96


In [41]:
from google.colab import files

shutil.make_archive('model', 'zip', '/content/crossview/model')
files.download('model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>